# VEERA — Kaggle GPU Deployment & Live Inference Server

**Failure-Aware Visual Evidence Acquisition for Reliable Multimodal AI**

This deployment notebook loads the fine-tuned **Qwen2.5-VL-3B-Instruct** vision-language model with the saved **VEERA LoRA adapter** on a Kaggle GPU accelerator and exposes a secure FastAPI service for your local React frontend.

---

### Request Flow Architecture:
```
React Frontend (Windows Laptop: http://localhost:5173)
      │
      ▼ (HTTPS POST /api/v1/inspect with image & question)
Secure Tunnel (Cloudflare Quick Tunnel / ngrok)
      │
      ▼
FastAPI Server (Kaggle GPU: http://127.0.0.1:8000)
      │
      ├──> Global Pass: Qwen2.5-VL-3B-Instruct + VEERA LoRA Adapter
      ├──> Reliability Evaluation: G(x, q) = |p_Yes - p_No|
      ├──> Threshold Check (tau = 0.7853)
      │      ├── G(x, q) >= tau ──> Confident Direct Output
      │      └── G(x, q) < tau  ──> Multi-Crop Acquisition & Reassessment
      │                               ├── Consensus >= 75% ──> Verified Verdict
      │                               └── Split / Ambiguous ──> Failure-Aware Abstention
      ▼
JSON Response (status, answer, reliability_score, latency, crops, reason)
```

> **Important Note on Domain Scope:** VEERA's fine-tuned weights were trained on the **RePOPE binary object-presence benchmark** (MS-COCO objects). They should not be represented as a certified or fully calibrated physical structural defect detector. Multi-crop evidence acquisition is an experimental research prototype.

## Cell A — Environment & Hardware Inspection
Verifies that an NVIDIA GPU accelerator is enabled and checks available VRAM.

In [ ]:
# Cell A: Hardware & Network Environment Inspection
import sys
import platform
import socket
import urllib.request
import urllib.error
import torch

print('=' * 75)
print('             VEERA DEPLOYMENT PRE-FLIGHT DIAGNOSTICS')
print('=' * 75)
print(f'Python Version  : {platform.python_version()}')
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')

# 1. Hardware Inspection
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'SUCCESS [GPU]   : {device_name} ({vram_gb:.2f} GB VRAM detected)')
    if vram_gb < 14:
        print('NOTE [GPU]      : Available VRAM is under 14 GB. Model will be loaded in float16 precision.')
else:
    print('\n' + '-' * 75)
    print('ERROR: NO GPU DETECTED!')
    print('Qwen2.5-VL-3B-Instruct requires an NVIDIA GPU accelerator.')
    print('To enable GPU in Kaggle:')
    print('  1. In the right-hand panel, expand Notebook options -> Accelerator.')
    print('  2. Select GPU T4 x2 or GPU P100.')
    print('  3. Re-run this cell.')
    print('-' * 75 + '\n')
    raise SystemExit('GPU accelerator is required to run VEERA neural inference.')

# 2. Lightweight Network & DNS Pre-flight Check
print('\nChecking Network & Hugging Face DNS connectivity...')

def verify_internet_connectivity(host='huggingface.co', timeout=5):
    # Step A: DNS Resolution
    try:
        ip = socket.gethostbyname(host)
    except socket.gaierror as e:
        return False, f"DNS resolution failed for '{host}': {e}."
    except Exception as e:
        return False, f"DNS resolution failed: {e}."

    # Step B: Lightweight HTTPS Probe
    try:
        url = f'https://{host}/api/models/Qwen/Qwen2.5-VL-3B-Instruct'
        req = urllib.request.Request(url, headers={'User-Agent': 'VEERA-Preflight/1.0'})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            if resp.status in (200, 301, 302):
                return True, f'Online: Reachable at {ip} (HTTP status: {resp.status}).'
            return True, f'Online (Status: {resp.status}).'
    except urllib.error.URLError as e:
        return False, f'HTTPS request failed: {e}.'
    except Exception as e:
        return False, f'Network check failed: {e}.'

online, net_msg = verify_internet_connectivity()

if online:
    print(f'SUCCESS [NET]   : {net_msg}')
    print('=' * 75 + '\n')
else:
    print('\n' + '=' * 75)
    print('ERROR: INTERNET ACCESS IS DISABLED IN THIS KAGGLE SESSION!')
    print(f'Reason: {net_msg}')
    print('\nThe base model weights cannot be downloaded because DNS and outbound HTTPS')
    print('traffic are blocked when Kaggle Internet is switched off.')
    print('\nEXACT FIX IN KAGGLE NOTEBOOK EDITOR (WEB UI):')
    print('  1. Look at the right-hand sidebar panel under Notebook options.')
    print('  2. Locate the Internet setting (currently set to Internet off).')
    print('  3. Click the toggle switch to change it to Internet on.')
    print('     (Note: Kaggle will prompt Enabling internet will restart your session).')
    print('  4. Click Turn on Internet to confirm.')
    print('  5. Once the session restarts, verify Accelerator is still GPU T4 x2.')
    print('  6. Re-run from Cell A onwards.')
    print('=' * 75 + '\n')
    raise SystemExit('Kaggle Internet must be enabled in Notebook options to download model weights.')


## Cell B — Dependency Installation
Installs compatible versions of FastAPI, Uvicorn, PEFT, Transformers, and Qwen-VL utilities.

In [ ]:
# Cell B: Install Dependencies
# Remove Kaggle's pre-installed incompatible torchao 0.10.0 that conflicts with newest PEFT
!pip uninstall -y -q torchao
!pip install -q "fastapi>=0.115.0" "uvicorn[standard]>=0.30.0" "python-multipart>=0.0.9" "transformers>=4.45.0" "peft>=0.13.0" "qwen-vl-utils>=0.0.8" "nest-asyncio>=1.6.0" "pyngrok>=7.0.0" "pillow>=10.4.0"

import transformers
import peft
import fastapi
import uvicorn

print(f"Transformers: {transformers.__version__}")
print(f"PEFT        : {peft.__version__}")
print(f"FastAPI     : {fastapi.__version__}")
print(f"Uvicorn     : {uvicorn.__version__}")
print("All required dependencies installed.")


## Cell C — Locate the Fine-Tuned LoRA Adapter
Detects the adapter files uploaded as a Kaggle dataset or in the working directory.

In [ ]:
# Cell C: Locate Adapter
import os
from pathlib import Path

# Search common Kaggle paths for the VEERA adapter
candidate_paths = [
    Path("/kaggle/input/veera-finetuned-release/adapter"),
    Path("/kaggle/input/veera-adapter/adapter"),
    Path("/kaggle/input/veera/VEERA_finetuned_release/adapter"),
    Path("/kaggle/working/VEERA/VEERA_finetuned_release/adapter"),
    Path("./VEERA_finetuned_release/adapter"),
    Path("./adapter"),
]

ADAPTER_DIR = None
for p in candidate_paths:
    if p.exists() and (p / "adapter_model.safetensors").exists():
        ADAPTER_DIR = p
        break

# Deep recursive search if not in default candidate paths
if ADAPTER_DIR is None:
    found = list(Path("/kaggle").rglob("adapter_model.safetensors"))
    if not found:
        found = list(Path(".").rglob("adapter_model.safetensors"))
    if found:
        ADAPTER_DIR = found[0].parent

if ADAPTER_DIR is None or not (ADAPTER_DIR / "adapter_model.safetensors").exists():
    print("""
    =========================================================================
    ❌ ERROR: LoRA adapter weights (adapter_model.safetensors) not found!
    
    To attach the adapter in Kaggle:
      1. In the top-right toolbar, click '+ Add Input'.
      2. Select 'Upload a dataset' and upload your 'VEERA_finetuned_release' folder.
      3. Once uploaded, re-run this cell.
    =========================================================================
    """)
    raise FileNotFoundError("adapter_model.safetensors not found in search paths.")

print(f"SUCCESS: Found VEERA LoRA Adapter directory: {ADAPTER_DIR}")
print("Artifact files present:")
for f in sorted(ADAPTER_DIR.iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / (1024**2)
        print(f"  - {f.name:<28} ({size_mb:6.2f} MB)")


## Cell D — Load Qwen2.5-VL-3B-Instruct Base Model & LoRA Adapter
Loads the base model in 16-bit precision (`torch.float16`) onto the GPU and attaches the fine-tuned PEFT adapter.

In [ ]:
# Cell D: Load Base Model and LoRA Adapter
import sys
import socket
import urllib.request
import urllib.error
import torch

# Prevent Kaggle's pre-installed torchao 0.10.0 from breaking PEFT LoRA dispatch
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

BASE_MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

# Pre-flight network check before triggering heavy Transformers download
try:
    socket.gethostbyname('huggingface.co')
except socket.gaierror as e:
    print('\n' + '=' * 75)
    print(f"ERROR: Cannot reach huggingface.co ('{e}').")
    print('Kaggle Internet is disabled in this session.')
    print('\nFIX: In the right sidebar -> Notebook options -> toggle Internet to ON.')
    print('Then restart session and re-run.')
    print('=' * 75 + '\n')
    raise SystemExit('Hugging Face is unreachable. Enable Internet in Kaggle Notebook options.')

print(f'1. Downloading/Loading base model: {BASE_MODEL_ID} (~6.2 GB weights)...')
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)

print(f'2. Attaching VEERA fine-tuned LoRA adapter from {ADAPTER_DIR}...')
# Guard against torchao dispatcher error in PEFT
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()

print('3. Loading vision-language processor...')
try:
    processor = AutoProcessor.from_pretrained(str(ADAPTER_DIR))
except Exception:
    processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

# Confirm adapter is attached and parameters are frozen for inference
print('\n--- Model Status Report ---')
print(f'Model Class          : {type(model).__name__}')
print(f'Active LoRA Adapters : {getattr(model, "active_adapters", ["default"])}')
print(f'Compute Device       : {next(model.parameters()).device}')
print(f'Weight Data Type     : {next(model.parameters()).dtype}')
print('SUCCESS: Base model and VEERA LoRA weights are active in GPU memory.')


## Cell E — Real Inference Smoke Test
Executes a single-pass inference on a sample image and computes the empirical reliability gap $G(x, q) = |p_{\text{Yes}} - p_{\text{No}}|$ using verified token IDs `9454` and `2753`.

In [ ]:
# Cell E: Real Neural Inference Smoke Test
import time
from PIL import Image, ImageDraw
import torch.nn.functional as F

# Create a synthetic test image with a dark horizontal bar
test_img = Image.new("RGB", (640, 480), color=(220, 220, 220))
draw = ImageDraw.Draw(test_img)
draw.rectangle([180, 220, 460, 260], fill=(20, 20, 20))
test_question = "Is there a dark line or mark on this surface?"

print(f"Querying: '{test_question}'")
start_time = time.perf_counter()

# Exact prompt phrasing from research notebook Cell 335
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": test_img},
        {"type": "text", "text": f"{test_question} Answer with only yes or no."},
    ],
}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
target_device = next(model.parameters()).device
inputs = {k: v.to(target_device) for k, v in inputs.items() if isinstance(v, torch.Tensor)}

with torch.inference_mode():
    outputs = model(**inputs)
    # Logits at the last prompt position predict the first generated answer token
    next_token_logits = outputs.logits[0, -1].float()
    probs = F.softmax(next_token_logits, dim=-1)
    
    generated = model.generate(**inputs, max_new_tokens=5, do_sample=False)

prompt_length = inputs["input_ids"].shape[1]
raw_text = processor.batch_decode(generated[:, prompt_length:], skip_special_tokens=True)[0].strip()

# Verified token IDs under Qwen2.5-VL tokenizer
TOKEN_ID_YES = 9454
TOKEN_ID_NO = 2753

p_yes = float(probs[TOKEN_ID_YES].item())
p_no = float(probs[TOKEN_ID_NO].item())
gap = abs(p_yes - p_no)
latency = time.perf_counter() - start_time

print("\n--- Smoke Test Results ---")
print(f"Raw Generated Text : '{raw_text}'")
print(f"p(Yes) [token 9454] : {p_yes:.4f} ({p_yes:.1%})")
print(f"p(No)  [token 2753] : {p_no:.4f} ({p_no:.1%})")
print(f"Reliability Gap G  : {gap:.4f} ({gap:.1%})")
print(f"Inference Latency  : {latency:.3f}s")
print(f"Validation Tau     : 0.7853 -> Gating: {'DIRECT PASS (Autonomous)' if gap >= 0.7853 else 'FLAGGED FOR CROPS'}")
print("SUCCESS: Genuine neural forward pass and probability gap calculation verified!")


## Cell F — FastAPI Inference Server
Runs the VEERA FastAPI backend asynchronously inside the Kaggle notebook.

In [ ]:
# Cell F: FastAPI Backend Server Definition (Full Multi-Round & Failure-Aware Policy)
import io
import time
import base64
import asyncio
import threading
import nest_asyncio
import traceback
import uuid
import json
from typing import List, Optional, Dict, Any
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from PIL import Image
import torch
import torch.nn.functional as F
from qwen_vl_utils import process_vision_info

nest_asyncio.apply()

# Reuse existing app instance if already defined to preserve background uvicorn binding
if 'app' not in globals() or app is None:
    app = FastAPI(title="VEERA Kaggle GPU Backend", version="1.1.0")
    app.add_middleware(
        CORSMiddleware,
        allow_origins=["*"],
        allow_credentials=True,
        allow_methods=["*"],
        allow_headers=["*"],
    )

DEFAULT_TAU = 0.7853
MAX_EVIDENCE_ROUNDS = 3

def normalize_yes_no(text):
    clean = text.strip().lower()
    first = clean.split()[0].strip(".,!?;:") if clean else ""
    if first in {"yes", "no"}: return first
    if "yes" in clean and "no" not in clean: return "yes"
    if "no" in clean and "yes" not in clean: return "no"
    return "unclear"

def evaluate_image_query(img, query):
    t0 = time.perf_counter()
    msg = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img.convert("RGB")},
            {"type": "text", "text": f"{query} Answer with only yes or no."},
        ],
    }]
    prompt = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(msg)
    inp = processor(
        text=[prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )
    dev = next(model.parameters()).device
    inp = {k: v.to(dev) for k, v in inp.items() if isinstance(v, torch.Tensor)}
    
    with torch.inference_mode():
        out = model(**inp)
        nxt = out.logits[0, -1].float()
        prb = F.softmax(nxt, dim=-1)
        gen = model.generate(**inp, max_new_tokens=5, do_sample=False)
    
    raw = processor.batch_decode(gen[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
    pyes = float(prb[9454].item())
    pno = float(prb[2753].item())
    gap = abs(pyes - pno)
    ans = normalize_yes_no(raw)
    if ans not in {"yes", "no"}:
        ans = "yes" if pyes >= pno else "no"
    return ans, gap, pyes, pno, raw, round(time.perf_counter() - t0, 3)

def generate_crops(img, crop_size=400):
    w, h = img.size
    if w <= crop_size or h <= crop_size:
        cw, ch = max(16, int(w * 0.65)), max(16, int(h * 0.65))
    else:
        cw, ch = crop_size, crop_size
    cw, ch = min(cw, w), min(ch, h)
    cx, cy = max(0, (w - cw) // 2), max(0, (h - ch) // 2)
    pos = [
        ("Top-Left Region", (0, 0, cw, ch)),
        ("Center ROI", (cx, cy, cx + cw, cy + ch)),
        ("Top-Right Region", (max(0, w - cw), 0, w, ch)),
        ("Bottom-Right Region", (max(0, w - cw), max(0, h - ch), w, h)),
    ]
    crops = []
    for idx, (lbl, (l, t, r, b)) in enumerate(pos, 1):
        sub = img.crop((l, t, r, b))
        buf = io.BytesIO()
        sub_thumb = sub.copy()
        sub_thumb.thumbnail((240, 240))
        sub_thumb.save(buf, format="JPEG", quality=75)
        b64 = f"data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode('utf-8')}"
        crops.append({
            "id": f"crop_{idx}",
            "label": lbl,
            "bbox": [t, l, b, r],
            "image_pil": sub,
            "image": b64,
        })
    return crops

def build_requested_evidence(q):
    lq = q.lower()
    if any(k in lq for k in ["crack", "fracture", "damage", "scratch", "break", "split"]):
        return [
            {"type": "close_up", "instruction": "Please upload a closer photograph of the suspected fissure near the surface centre.", "rationale": "High optical detail is required to distinguish fine fractures from dust."},
            {"type": "alternate_angle", "instruction": "Please photograph the surface from an oblique angle with reduced shadow/glare.", "rationale": "Grazing-angle lighting reveals physical surface relief."},
            {"type": "wider_view", "instruction": "Please upload a wider photograph showing the surrounding wall/surface continuation.", "rationale": "Context is needed to trace structural boundary lines."}
        ]
    return [
        {"type": "close_up", "instruction": "Please upload a closer photograph of the specific region of interest.", "rationale": "Current viewpoint lacks fine visual detail."},
        {"type": "alternate_angle", "instruction": "Please provide an alternate angle under clear, diffuse lighting.", "rationale": "Eliminates ambiguous reflections and shadows."}
    ]

@app.get("/health")
def health():
    dev = str(next(model.parameters()).device)
    details = {"device": dev}
    if torch.cuda.is_available():
        details["gpu_name"] = torch.cuda.get_device_name(0)
        details["allocated_mb"] = round(torch.cuda.memory_allocated(0)/(1024*1024), 2)
    return {
        "status": "ok",
        "version": "1.1.0",
        "device": dev,
        "base_model_id": BASE_MODEL_ID,
        "lora_adapter_present": True,
        "lora_adapter_loaded": True,
        "threshold_tau": DEFAULT_TAU,
        "device_details": details,
    }

async def process_pipeline(image_file, question, threshold, round_num=1, view_type="initial", previous_history=None, session_id=None):
    t_start = time.perf_counter()
    clean_q = question.strip()
    if not clean_q:
        raise HTTPException(status_code=422, detail="Question cannot be empty.")
    content = await image_file.read()
    try:
        pil_img = Image.open(io.BytesIO(content)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image file: {str(e)}")
    
    tau = threshold if (threshold and 0.0 < threshold < 1.0) else DEFAULT_TAU
    sess_id = session_id or str(uuid.uuid4())[:8]
    history = []
    if previous_history:
        try:
            history = json.loads(previous_history) if isinstance(previous_history, str) else list(previous_history)
        except Exception:
            pass
    
    # 1. Global pass
    t_g0 = time.perf_counter()
    ans, gap, pyes, pno, raw, lat_g = evaluate_image_query(pil_img, clean_q)
    
    if gap >= tau:
        tot_lat = round(time.perf_counter() - t_start, 3)
        history.append({
            "round_number": round_num, "view_type": view_type, "answer": ans,
            "reliability_score": round(gap, 4), "status": "answered",
            "reason": f"Global pass confident (G={gap:.2%} >= tau={tau:.2%})."
        })
        return {
            "status": "answered", "answer": ans, "reliability_score": round(gap, 4),
            "evidence_inspections": 0, "latency_seconds": tot_lat,
            "reason": f"Global pass yielded confident reliability gap G(x, q) = {gap:.2%} exceeding threshold tau = {tau:.2%}.",
            "crops_inspected": [],
            "findings": [{
                "category": "Global Autonomous Pass",
                "description": f"Model answered '{ans.upper()}' with raw output '{raw}'.",
                "evidence": f"p(Yes)={pyes:.3f}, p(No)={pno:.3f}, gap={gap:.3f}",
            }],
            "verification": {"performed": False, "initial_assessment": f"High certainty ({gap:.2%}). Verdict: {ans.upper()}.", "additional_evidence_acquired": False},
            "question": clean_q, "assessment": "defect_detected" if ans == "yes" else "no_defect",
            "summary": f"Analysis completed autonomously with high reliability ({gap:.2%}).",
            "initial_prediction": ans, "initial_gap": round(gap, 4), "threshold": tau,
            "additional_inspection_performed": False,
            "evidence_sufficiency": {"sufficient": True, "reason": "Reliability gap exceeds validation threshold."},
            "requested_evidence": [], "next_action": None, "evidence_history": history,
            "session_id": sess_id, "round_number": round_num,
            "latency_breakdown": {"initial_inference_seconds": lat_g, "crop_inspection_seconds": 0.0, "total_seconds": tot_lat},
            "device_used": str(next(model.parameters()).device)
        }
    
    # 2. Multi-crop pass
    t_c0 = time.perf_counter()
    raw_crops = generate_crops(pil_img)
    crop_results = []
    crop_votes = []
    for c in raw_crops:
        c_ans, c_gap, _, _, _, _ = evaluate_image_query(c["image_pil"], clean_q)
        crop_results.append({
            "id": c["id"], "label": c["label"], "bbox": c["bbox"],
            "vote": c_ans, "reliability_score": round(c_gap, 4),
            "notes": f"Local gap: {c_gap:.2%}, vote: {c_ans.upper()}",
            "image": c["image"],
        })
        crop_votes.append(c_ans)
    lat_crops = round(time.perf_counter() - t_c0, 3)
    tot_lat = round(time.perf_counter() - t_start, 3)
    
    yes_cnt = crop_votes.count("yes")
    no_cnt = crop_votes.count("no")
    mean_crop_gap = sum(c["reliability_score"] for c in crop_results) / len(crop_results)
    consensus_cnt = max(yes_cnt, no_cnt)
    consensus_ratio = consensus_cnt / len(crop_results)
    majority_vote = "yes" if yes_cnt > no_cnt else ("no" if no_cnt > yes_cnt else "tie")
    
    # Failure-aware decision logic:
    # Contradiction (yes > 0 and no > 0) with low reliability (< tau) -> MUST ABSTAIN
    is_contradictory = (yes_cnt > 0 and no_cnt > 0 and mean_crop_gap < tau)
    is_tied = (yes_cnt == no_cnt)
    is_weak = (mean_crop_gap < (tau * 0.5))
    is_insufficient = is_contradictory or is_tied or is_weak or (consensus_ratio < 0.75)
    
    if is_insufficient:
        disp_status = "review"
        disp_answer = "uncertain"
        disp_assess = "inconclusive"
        if is_tied:
            reason_txt = f"Crop inspection deadlock ({yes_cnt} Yes vs {no_cnt} No). Abstaining to prevent false certainty."
        elif is_contradictory:
            reason_txt = f"Contradictory evidence across localized crops ({yes_cnt} Yes vs {no_cnt} No). Low mean gap (G={mean_crop_gap:.2%} < tau={tau:.2%}) precludes safe majority override."
        else:
            reason_txt = f"Crop evaluation yielded weak evidence (mean gap G={mean_crop_gap:.2%} < tau={tau:.2%})."
        
        if round_num >= MAX_EVIDENCE_ROUNDS:
            next_act = "recommend_human_inspection"
            reason_txt += " Maximum automated rounds reached. Professional physical inspection recommended."
            req_items = []
        else:
            next_act = "upload_additional_evidence"
            req_items = build_requested_evidence(clean_q)
    else:
        disp_status = "answered"
        disp_answer = majority_vote
        disp_assess = "defect_detected" if majority_vote == "yes" else "no_defect"
        reason_txt = f"Multi-crop inspection resolved ambiguity with {consensus_cnt}/4 consensus ({majority_vote.upper()}) and mean gap {mean_crop_gap:.2%}."
        next_act = None
        req_items = []
    
    history.append({
        "round_number": round_num, "view_type": view_type, "answer": disp_answer,
        "reliability_score": round(mean_crop_gap, 4), "status": disp_status, "reason": reason_txt
    })
    
    primary_roi = crop_results[0]["image"] if crop_results else None
    return {
        "status": disp_status, "answer": disp_answer, "reliability_score": round(mean_crop_gap, 4),
        "evidence_inspections": len(crop_results), "latency_seconds": tot_lat, "reason": reason_txt,
        "crops_inspected": crop_results,
        "findings": [{
            "category": "Multi-Crop Verification [Failure-Aware Policy]",
            "description": reason_txt,
            "evidence": f"Inspected {len(crop_results)} crops. Consensus: {consensus_ratio:.0%}.",
            "region_image": primary_roi,
        }],
        "verification": {
            "performed": True,
            "initial_assessment": f"Initial pass ambiguous (G={gap:.2%}).",
            "reassessment": reason_txt,
            "additional_evidence_acquired": True
        },
        "question": clean_q, "assessment": disp_assess,
        "summary": "Abstention Policy Triggered: Inconclusive Evidence." if is_insufficient else f"Adaptive Verification Succeeded: {disp_answer.upper()}.",
        "initial_prediction": ans, "initial_gap": round(gap, 4), "threshold": tau,
        "additional_inspection_performed": True,
        "evidence_sufficiency": {"sufficient": not is_insufficient, "reason": reason_txt},
        "requested_evidence": req_items, "next_action": next_act,
        "evidence_history": history, "session_id": sess_id, "round_number": round_num,
        "latency_breakdown": {"initial_inference_seconds": lat_g, "crop_inspection_seconds": lat_crops, "total_seconds": tot_lat},
        "device_used": str(next(model.parameters()).device)
    }

@app.post("/api/v1/inspect")
async def inspect(image: UploadFile = File(...), question: str = Form(...), threshold: Optional[float] = Form(DEFAULT_TAU), session_id: Optional[str] = Form(None), round_number: Optional[int] = Form(1), view_type: Optional[str] = Form("initial"), previous_history: Optional[str] = Form(None)):
    try:
        return await process_pipeline(image, question, threshold, round_num=round_number or 1, view_type=view_type or "initial", previous_history=previous_history, session_id=session_id)
    except Exception as exc:
        tb = traceback.format_exc()
        print(f"\n[ERROR /api/v1/inspect]:\n{tb}")
        raise HTTPException(status_code=500, detail=f"Inference error: {str(exc)}")

@app.post("/api/v1/inspect/follow-up")
async def inspect_follow_up(image: UploadFile = File(...), question: str = Form(...), session_id: str = Form(...), round_number: int = Form(2), view_type: str = Form("close_up"), threshold: Optional[float] = Form(DEFAULT_TAU), previous_history: Optional[str] = Form(None)):
    try:
        return await process_pipeline(image, question, threshold, round_num=round_number, view_type=view_type, previous_history=previous_history, session_id=session_id)
    except Exception as exc:
        tb = traceback.format_exc()
        print(f"\n[ERROR /api/v1/inspect/follow-up]:\n{tb}")
        raise HTTPException(status_code=500, detail=f"Follow-up error: {str(exc)}")

print("SUCCESS: Full Multi-Round Failure-Aware FastAPI backend ready.")


## Cell G — Start Uvicorn Server in Background Thread
Launches the FastAPI server on `http://127.0.0.1:8000` inside the notebook session.

In [ ]:
# Cell G: Launch FastAPI Server in Dedicated Background Event Loop
import uvicorn
import threading
import time
import asyncio
import urllib.request
import json

server_config = uvicorn.Config(app=app, host="127.0.0.1", port=8000, log_level="info")

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    srv = uvicorn.Server(server_config)
    loop.run_until_complete(srv.serve())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# Verify /health endpoint locally inside Kaggle
try:
    req = urllib.request.urlopen("http://127.0.0.1:8000/health")
    health_data = json.loads(req.read().decode("utf-8"))
    print(f"SUCCESS: Server is running! Health check response:")
    print(json.dumps(health_data, indent=2))
except Exception as e:
    print(f"ERROR connecting to local health endpoint: {e}")


## Cell H — Secure HTTPS Tunnel to Expose API to Frontend
Uses **Cloudflare Quick Tunnel (recommended — instant, no account or tokens required)** to generate a secure public HTTPS endpoint.

In [ ]:
# Cell H: Expose API via Secure Cloudflare Tunnel (No account or tokens required)
import subprocess
import time
import os
import re

# Download cloudflared binary if not present
!wget -nc -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

# Remove any stale logfile
if os.path.exists("/tmp/cloudflared.log"):
    os.remove("/tmp/cloudflared.log")

# Start tunnel in background with explicit logfile
tunnel_proc = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--logfile", "/tmp/cloudflared.log"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

public_url = None
print("Waiting for Cloudflare Tunnel to allocate public HTTPS endpoint...")
for _ in range(35):
    time.sleep(1)
    if os.path.exists("/tmp/cloudflared.log"):
        with open("/tmp/cloudflared.log", "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    print("=" * 75)
    print(f"\n🚀 LIVE PUBLIC HTTPS API ENDPOINT:\n   {public_url}\n")
    print("=" * 75)
    print("\n👉 COPY TO YOUR WINDOWS LAPTOP .env FILE:")
    print(f"VITE_API_BASE_URL={public_url}")
    print("VITE_MOCK_MODE=false\n")
    print("Then restart your frontend (npm run dev) on your laptop to inspect real images!")
else:
    print("Cloudflare tunnel URL not captured yet. Check /tmp/cloudflared.log or use the ngrok alternative below.")


### Alternative Tunnel: ngrok (Optional)
If you prefer ngrok, add your `NGROK_AUTHTOKEN` in Kaggle Secrets (Add-ons -> Secrets) and run this cell.

In [ ]:
# Optional: ngrok tunnel alternative
from pyngrok import ngrok
import os

try:
    from kaggle_secrets import UserSecretsClient
    ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
except Exception:
    ngrok_token = os.environ.get("NGROK_AUTHTOKEN")

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(8000)
    print(f"ngrok Live URL: {tunnel.public_url}")
    print(f"Set in laptop .env: VITE_API_BASE_URL={tunnel.public_url}")
else:
    print("No NGROK_AUTHTOKEN found in Kaggle Secrets. Cloudflare Tunnel above is active.")
